In [3]:
# في هذه الخلية نحمّل المكتبات التي سنستخدمها في قراءة البيانات وتنظيفها وتدريب المودل وحفظه

import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import joblib


In [4]:
# في هذه الخلية نقرأ ملف الإكسل الجديد ونحمل كل الشيتات

file_path = "trainData_copy_auto_fixed.xlsx"
all_sheets = pd.read_excel(file_path, sheet_name=None)

print("Sheet names:", list(all_sheets.keys()))


Sheet names: ['Gemini', 'ChatGPT', 'Tele', 'Telegram', 'DeepSeek']


In [5]:
# في هذه الخلية نجمع الشيتات كلها في جدول واحد حتى ندرّب على كامل البيانات

dfs = []
for sheet_name, sheet_df in all_sheets.items():
    if sheet_df is None or sheet_df.empty:
        continue
    
    temp = sheet_df.copy()
    temp["source"] = sheet_name
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])
df.head(8)


Total rows: 2321
Total columns: 8


,نص الاستشارة,التصنيف,final_text,التصنيف_المصحح_تلقائيا,نقاط_اسريه,نقاط_احوال_شخصيه,التصنيف_النهائي,source
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية,شريكي سحب سيوله من المؤسسه بدون فواتير وش الحل؟,NaN,NaN,NaN,القضايا التجارية,Gemini
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية,صاحب العمل فصلني بدون سابق انذار ولا اعطاني مك...,NaN,NaN,NaN,القضايا العمالية,Gemini
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية,تعرضت لابتزاز بصور خاصه من حساب وهمي في سناب,NaN,NaN,NaN,القضايا الجنائية,Gemini
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,القضايا الأسرية,زوجي هجر البيت ولا يصرف علي العيال من ٤ شهور,القضايا الأسرية,9.0,3.0,القضايا الأسرية,Gemini
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية,شريت شقه وطلعت فيها عيوب في السباكه والمالك ير...,NaN,NaN,NaN,القضايا العقارية,Gemini
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,القضايا الإدارية,تم استبعادي من مسابقه وظيفيه حكوميه رغم انطباق...,NaN,NaN,NaN,القضايا الإدارية,Gemini
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,القضايا المالية,البنك سحب مبلغ اكبر من القسط الشهري المتفق عليه,NaN,NaN,NaN,القضايا المالية,Gemini
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية,اخوي الكبير رافض يوزع ورث ابوي ومستولي علي الم...,قضايا الأحوال الشخصية,0.0,1.0,قضايا الأحوال الشخصية,Gemini


In [6]:
# في هذه الخلية نحدد أسماء الأعمدة
# إذا كانت أسماء الأعمدة عندك مختلفة عدلي هذه المتغيرات فقط

TEXT_COL = "نص الاستشارة"
LABEL_COL = "التصنيف_النهائي"

print("Text column:", TEXT_COL)
print("Label column:", LABEL_COL)
print("Available columns:", list(df.columns))


Text column: نص الاستشارة
Label column: التصنيف_النهائي
Available columns: ['نص الاستشارة', 'التصنيف', 'final_text', 'التصنيف_المصحح_تلقائيا', 'نقاط_اسريه', 'نقاط_احوال_شخصيه', 'التصنيف_النهائي', 'source']


In [7]:
# في هذه الخلية نتأكد أن النص والتصنيف موجودين ونحذف الصفوف التي فيها نقص

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

df = df[(df[TEXT_COL] != "") & (df[LABEL_COL] != "")]
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)

print("Rows after cleaning:", df.shape[0])
df[[TEXT_COL, LABEL_COL, "source"]].head(5)


Rows after cleaning: 2321


,نص الاستشارة,التصنيف_النهائي,source
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية,Gemini
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية,Gemini
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية,Gemini
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,القضايا الأسرية,Gemini
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية,Gemini


In [8]:
import re

AR_STOPWORDS = set("""
 ورحمة الله وبركاته من في على الى إلى عن مع بين عند لدى عندي لدي له لها لهم هن هو هي انا انت انتي نحن هم هذا هذه ذلك تلك السلام عليكم 
كان تكون يكون كانت كنت جدا فقط ايضا ثم حيث اذا لأن لان قد لقد لا لم لن ما ماذا كيف ليش وش كل بعض اكثر اقل
 مرة مرات قبل بعد خلال حول حتى بدون فوق تحت داخل خارج رفع دعوى اريد 
""".split())

def normalize_text_ar(text: str) -> str:
    text = str(text)

    # حذف روابط وإيميلات
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)

    # إزالة التشكيل
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)

    # توحيد أحرف
    text = text.replace("أ","ا").replace("إ","ا").replace("آ","ا")
    text = text.replace("ى","ي").replace("ة","ه")

    # حذف الترقيم العربي والانجليزي مثل ؟ ، ؛ … . , !
    text = re.sub(r"[؟،؛…!\"'#\$%&\(\)\*\+,\-\.\/:;<=>@\[\]\\\^_`{\|}~]", " ", text)

    # إبقاء الحروف العربية والمسافات فقط
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # تقليل تكرار الحروف
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # تنظيف مسافات
    text = re.sub(r"\s+", " ", text).strip()

    # حذف كلمات الوقف والكلمات القصيرة جدا
    tokens = text.split()
    tokens = [w for w in tokens if w not in AR_STOPWORDS and len(w) > 2]

    return " ".join(tokens)

df["final_text"] = df[TEXT_COL].apply(normalize_text_ar)

df[[TEXT_COL, "final_text"]].head(8)


,نص الاستشارة,final_text
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,شريكي سحب سيوله المؤسسه فواتير الحل
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,صاحب العمل فصلني سابق انذار ولا اعطاني مكافاه
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,تعرضت لابتزاز بصور خاصه حساب وهمي سناب
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,زوجي هجر البيت ولا يصرف علي العيال شهور
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,شريت شقه وطلعت فيها عيوب السباكه والمالك يرفض ...
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,استبعادي مسابقه وظيفيه حكوميه رغم انطباق الشروط
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,البنك سحب مبلغ اكبر القسط الشهري المتفق عليه
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,اخوي الكبير رافض يوزع ورث ابوي ومستولي علي الم...


In [9]:
# في هذه الخلية نشوف عدد الحالات في كل تصنيف
# هذا يساعدنا نتأكد أن البيانات متوازنة بشكل معقول

counts = df[LABEL_COL].value_counts()
print(counts)

print("Number of classes:", counts.shape[0])


التصنيف_النهائي
القضايا العمالية         458
القضايا العقارية         327
القضايا الإدارية         291
القضايا التجارية         278
قضايا الأحوال الشخصية    278
القضايا الجنائية         271
القضايا المالية          259
القضايا الأسرية          159
Name: count, dtype: int64
Number of classes: 8


In [10]:
# في هذه الخلية نقسم البيانات إلى تدريب واختبار
# نحافظ على نفس توزيع الفئات باستخدام stratify

X_text = df["final_text"].fillna("").astype(str)
y_text = df[LABEL_COL].astype(str)

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y_text,
    test_size=0.2,
    random_state=42,
    stratify=y_text
)

print("Train size:", len(X_train_text))
print("Test size:", len(X_test_text))


Train size: 1856
Test size: 465


In [11]:
# في هذه الخلية ندرب مودل سريع كخطوة أولى
# هذا يساعدنا نتأكد أن كل شيء شغال قبل نبدأ تحسينات GridSearch

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), max_features=20000, min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC(C=0.5, class_weight="balanced"))
])

baseline_model.fit(X_train_text, y_train_text)

y_pred_base = baseline_model.predict(X_test_text)

print("Baseline Accuracy:", accuracy_score(y_test_text, y_pred_base))
print("\nBaseline Report:\n")
print(classification_report(y_test_text, y_pred_base))


Baseline Accuracy: 0.8774193548387097

Baseline Report:

                       precision    recall  f1-score   support

      القضايا الأسرية       0.76      0.78      0.77        32
     القضايا الإدارية       0.87      0.83      0.85        58
     القضايا التجارية       0.87      0.80      0.83        56
     القضايا الجنائية       0.96      0.89      0.92        54
     القضايا العقارية       0.90      0.95      0.93        65
     القضايا العمالية       0.99      0.98      0.98        92
      القضايا المالية       0.71      0.88      0.79        52
قضايا الأحوال الشخصية       0.88      0.79      0.83        56

             accuracy                           0.88       465
            macro avg       0.87      0.86      0.86       465
         weighted avg       0.88      0.88      0.88       465



In [12]:
# في هذه الخلية نجرب أكثر من إعداد ونختار أفضل واحد بناء على f1 macro
# قد يأخذ وقت حسب الجهاز لكن بياناتك أقل من 2000 إلى 2500 فغالبا مقبول

pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC(class_weight="balanced"))
])

param_grid = [
    {
        "tfidf__analyzer": ["char_wb"],
        "tfidf__ngram_range": [(3,5), (4,6)],
        "tfidf__max_features": [12000, 20000],
        "tfidf__min_df": [1, 2],
        "tfidf__sublinear_tf": [True],
        "clf__C": [0.25, 0.5, 1, 2],
    },
    {
        "tfidf__analyzer": ["word"],
        "tfidf__ngram_range": [(1,2)],
        "tfidf__max_features": [5000, 8000, 12000],
        "tfidf__min_df": [1, 2],
        "tfidf__sublinear_tf": [True],
        "clf__C": [0.25, 0.5, 1, 2],
    }
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train_text, y_train_text)

print("Best params:", grid.best_params_)
print("Best CV f1_macro:", grid.best_score_)

best_model = grid.best_estimator_


Fitting 5 folds for each of 56 candidates, totalling 280 fits
Best params: {'clf__C': 2, 'tfidf__analyzer': 'char_wb', 'tfidf__max_features': 20000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (3, 5), 'tfidf__sublinear_tf': True}
Best CV f1_macro: 0.8612674043067605


In [13]:
# في هذه الخلية نقيم المودل الأفضل على test ونطبع التقرير

y_pred = best_model.predict(X_test_text)

print("Test Accuracy:", accuracy_score(y_test_text, y_pred))
print("\nTest Report:\n")
print(classification_report(y_test_text, y_pred))


Test Accuracy: 0.8774193548387097

Test Report:

                       precision    recall  f1-score   support

      القضايا الأسرية       0.76      0.78      0.77        32
     القضايا الإدارية       0.87      0.79      0.83        58
     القضايا التجارية       0.85      0.84      0.85        56
     القضايا الجنائية       0.96      0.89      0.92        54
     القضايا العقارية       0.90      0.97      0.93        65
     القضايا العمالية       0.99      0.98      0.98        92
      القضايا المالية       0.74      0.87      0.80        52
قضايا الأحوال الشخصية       0.85      0.79      0.81        56

             accuracy                           0.88       465
            macro avg       0.86      0.86      0.86       465
         weighted avg       0.88      0.88      0.88       465



In [14]:
from sklearn.metrics import confusion_matrix

FAMILY = "القضايا الأسرية"
PERSONAL = "قضايا الأحوال الشخصية"

labels = [FAMILY, PERSONAL]   # ترتيب ثابت وواضح

cm2 = confusion_matrix(y_test_text, y_pred, labels=labels)

print("أسرية → أحوال شخصية:", cm2[0][1])
print("أحوال شخصية → أسرية:", cm2[1][0])


أسرية → أحوال شخصية: 5
أحوال شخصية → أسرية: 6
